# LTX-2 Video+Audio Generation on AWS Trainium

This notebook demonstrates how to compile and run the [Lightricks LTX-2](https://huggingface.co/Lightricks/LTX-2) 19B-parameter audio-video diffusion model on AWS Trainium using NxDI (NeuronX Distributed Inference).

**What gets compiled for Neuron:**
- DiT transformer backbone (48 blocks, ~6B params) → Neuron TP=4
- Gemma 3-12B text encoder → Neuron TP=4

**What stays on CPU:**
- Video + Audio VAE decoders (~1B params)
- Tokenizer, scheduler, vocoder

**Hardware:** trn2.3xlarge (1 NeuronDevice, 4 logical NeuronCores with LNC=2)

**Performance:**
| Metric | Neuron (trn2) | GPU (g5.12xlarge) |
|--------|--------------|-------------------|
| Generation (CFG, 8 steps) | ~66s | ~48s |
| Hardware cost/hr | ~$0.90 (spot) | ~$5.67 |

## 0. Prerequisites

Run this notebook on a **trn2.3xlarge** instance with the Deep Learning AMI Neuron (Ubuntu 24.04).

```bash
# Activate the pre-installed Neuron environment
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate

# Install diffusers (LTX-2 requires dev version)
pip install git+https://github.com/huggingface/diffusers.git
pip install imageio imageio-ffmpeg ipywidgets

# Upload this package to the instance
# scp -r ltx2-neuron/ ubuntu@<instance-ip>:/home/ubuntu/
```

In [ ]:
import gc
import json
import os
import sys
import time

import torch

# Environment variables for Neuron compiler/runtime
os.environ.setdefault("NEURON_FUSE_SOFTMAX", "1")
os.environ.setdefault("NEURON_CUSTOM_SILU", "1")
os.environ.setdefault("NEURON_RT_STOCHASTIC_ROUNDING_EN", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Add the src directory to path
# When running as a notebook, use the notebook's directory; when running via nbconvert, use cwd
try:
    _nb_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _nb_dir = os.getcwd()
PACKAGE_DIR = os.path.abspath(os.path.join(_nb_dir, "..", "src"))
if not os.path.isdir(PACKAGE_DIR):
    PACKAGE_DIR = os.path.abspath(os.path.join(_nb_dir, "src"))
sys.path.insert(0, PACKAGE_DIR)
print(f"Package dir: {PACKAGE_DIR}")

print(f"PyTorch: {torch.__version__}")
try:
    import torch_neuronx
    print(f"torch-neuronx: {torch_neuronx.__version__}")
except ImportError:
    print("WARNING: torch_neuronx not found. Are you on a Neuron instance?")

## 1. Configuration

Set the video resolution, frame count, and output directories.

In [ ]:
# Video generation settings
HEIGHT, WIDTH, NUM_FRAMES = 384, 512, 25
NUM_STEPS = 8  # Distilled model uses 8 steps
TP_DEGREE = 4  # Tensor parallelism across 4 NeuronCores
SEED = 42

PROMPT = (
    "A golden retriever puppy runs across a sunny green meadow, "
    "its ears flapping in the wind. The camera follows from a low angle. "
    "Birds chirp in the background."
)

# Compiled model directories (persist across reboots if on /home/ubuntu/)
DIT_COMPILE_DIR = "/home/ubuntu/ltx2_nxdi_compiled_1024/"
GEMMA3_COMPILE_DIR = "/home/ubuntu/gemma3_encoder_compiled_1024/"
GEMMA3_SHARDED_DIR = "/home/ubuntu/gemma3_encoder_sharded/"
OUTPUT_DIR = "/home/ubuntu/ltx2_output/"

print(f"Video: {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_STEPS} steps")
print(f"TP degree: {TP_DEGREE}")
print(f"Prompt: {PROMPT[:80]}...")

## 2. Compile the DiT Transformer Backbone

The DiT backbone is the denoising workhorse — 48 transformer blocks that run 8 times per generation (once per denoising step), doubled with CFG. This is the component that benefits most from Neuron acceleration.

Compilation traces the model with NxDI's SPMD infrastructure and produces a single `model.pt` NEFF file. **First compilation takes ~2 minutes; subsequent runs load from cache.**

In [ ]:
from modeling_ltx2 import (
    LTX2BackboneInferenceConfig,
    NeuronLTX2BackboneApplication,
    replace_sdpa_with_bmm,
)
from neuronx_distributed_inference.models.config import NeuronConfig

# Replace SDPA with BMM for Neuron XLA compatibility
replace_sdpa_with_bmm()

# Load transformer config from HuggingFace
from huggingface_hub import hf_hub_download
config_path = hf_hub_download("Lightricks/LTX-2", "transformer/config.json")
with open(config_path) as f:
    hf_config = json.load(f)

# Compute latent dimensions
num_heads = hf_config["num_attention_heads"]
head_dim = hf_config["attention_head_dim"]
inner_dim = num_heads * head_dim
audio_num_heads = hf_config["audio_num_attention_heads"]
audio_head_dim = hf_config["audio_attention_head_dim"]
audio_inner_dim = audio_num_heads * audio_head_dim
audio_ca_dim = hf_config.get("audio_cross_attention_dim", audio_inner_dim)

latent_num_frames = (NUM_FRAMES - 1) // 8 + 1
latent_height = HEIGHT // 32
latent_width = WIDTH // 32
video_seq = latent_num_frames * latent_height * latent_width
audio_num_frames = round((NUM_FRAMES / 24.0) * 24.97)

backbone_neuron_config = NeuronConfig(
    tp_degree=TP_DEGREE,
    world_size=TP_DEGREE,
    torch_dtype=torch.bfloat16,
)

config = LTX2BackboneInferenceConfig(
    neuron_config=backbone_neuron_config,
    num_layers=hf_config["num_layers"],
    num_attention_heads=num_heads,
    attention_head_dim=head_dim,
    inner_dim=inner_dim,
    audio_num_attention_heads=audio_num_heads,
    audio_attention_head_dim=audio_head_dim,
    audio_inner_dim=audio_inner_dim,
    audio_cross_attention_dim=audio_ca_dim,
    caption_channels=hf_config.get("caption_channels", 3840),
    video_seq=video_seq,
    audio_seq=audio_num_frames,
    text_seq=1024,
    height=HEIGHT,
    width=WIDTH,
    num_frames=NUM_FRAMES,
)
config.hf_config_dict = hf_config

print(f"DiT: {hf_config['num_layers']} blocks, inner_dim={inner_dim}, audio_inner_dim={audio_inner_dim}")
print(f"Video: {video_seq} tokens, Audio: {audio_num_frames} tokens, Text: 1024 tokens")

In [ ]:
# Create the NxDI backbone application
from huggingface_hub import snapshot_download

local_transformer_path = snapshot_download(
    "Lightricks/LTX-2", allow_patterns=["transformer/*"]
)
local_transformer_path = os.path.join(local_transformer_path, "transformer")

backbone_app = NeuronLTX2BackboneApplication(
    model_path=local_transformer_path,
    config=config,
)

# Compile if not already cached
compiled_model_file = os.path.join(DIT_COMPILE_DIR, "model.pt")
if os.path.exists(compiled_model_file):
    size_gb = os.path.getsize(compiled_model_file) / 1e9
    print(f"Found cached DiT compiled model: {compiled_model_file} ({size_gb:.2f} GB)")
else:
    print(f"Compiling DiT backbone to {DIT_COMPILE_DIR} (first run, ~2 min)...")
    t0 = time.time()
    os.makedirs(DIT_COMPILE_DIR, exist_ok=True)
    backbone_app.compile(DIT_COMPILE_DIR)
    print(f"Compiled in {time.time() - t0:.1f}s")

## 3. Compile the Gemma 3-12B Text Encoder

LTX-2 uses Gemma 3-12B as its text encoder. We compile a custom encoder that outputs all 49 hidden states (embedding + 48 layers), which LTX-2 stacks and projects into text embeddings.

**First compilation takes ~3 minutes.** We also pre-shard the weights to disk (~5.5 GB per rank) so loading only requires ~6 GB peak memory instead of ~24 GB.

In [ ]:
# Check if Gemma3 encoder is already compiled
gemma3_compiled = os.path.exists(os.path.join(GEMMA3_COMPILE_DIR, "tp_0.pt"))
gemma3_sharded = os.path.exists(os.path.join(GEMMA3_SHARDED_DIR, "rank_0.pt"))

if gemma3_compiled:
    size_gb = os.path.getsize(os.path.join(GEMMA3_COMPILE_DIR, "tp_0.pt")) / 1e9
    print(f"Found cached Gemma3 compiled encoder: {GEMMA3_COMPILE_DIR} ({size_gb:.1f} GB per rank)")
else:
    print("Gemma3 encoder not compiled yet.")
    print("Run: python ../src/compile_gemma3.py")
    print("This takes ~3 minutes and produces tp_0.pt through tp_3.pt")

if gemma3_sharded:
    size_gb = os.path.getsize(os.path.join(GEMMA3_SHARDED_DIR, "rank_0.pt")) / 1e9
    print(f"Found pre-sharded Gemma3 weights: {GEMMA3_SHARDED_DIR} ({size_gb:.1f} GB per rank)")
else:
    print("Gemma3 weights not pre-sharded yet.")
    print("Run: python ../src/shard_gemma3_weights.py")
    print("This takes ~2 minutes and produces rank_0.pt through rank_3.pt")

## 4. Load the Diffusers Pipeline

We load the full Diffusers LTX2Pipeline first (text encoder, transformer, VAEs, vocoder all on CPU), then swap the text encoder and transformer with Neuron-compiled versions.

In [ ]:
print("Loading Diffusers LTX2Pipeline (CPU)...")
t0 = time.time()
from diffusers import LTX2Pipeline

pipe = LTX2Pipeline.from_pretrained(
    "Lightricks/LTX-2",
    torch_dtype=torch.bfloat16,
)
print(f"Pipeline loaded in {time.time() - t0:.1f}s")
print(f"Components: {', '.join(k for k, v in pipe.components.items() if v is not None)}")

## 5. Load DiT Backbone onto Neuron

Load the compiled NEFF, shard the transformer weights across 4 NeuronCores, and warm up.

In [ ]:
from pipeline import NeuronTransformerWrapper

print(f"Loading DiT backbone from {DIT_COMPILE_DIR}...")
cpu_transformer = pipe.transformer

t0 = time.time()
backbone_app.load(DIT_COMPILE_DIR)
print(f"DiT loaded in {time.time() - t0:.1f}s")

# Swap the pipeline's transformer with the Neuron wrapper
wrapper = NeuronTransformerWrapper(
    compiled_backbone=backbone_app,
    cpu_transformer=cpu_transformer,
    text_seq=1024,
)

# Free heavy transformer blocks (preprocessing layers are kept via wrapper refs)
del cpu_transformer.transformer_blocks
del cpu_transformer.norm_out, cpu_transformer.proj_out
del cpu_transformer.audio_norm_out, cpu_transformer.audio_proj_out
gc.collect()

pipe.transformer = wrapper
print("DiT transformer swapped to Neuron")

## 6. Load Gemma3 Text Encoder onto Neuron

Swap the CPU text encoder with the Neuron-compiled Gemma3 encoder using pre-sharded per-rank weights.

In [ ]:
import torch_neuronx
from neuronx_distributed.trace.trace import (
    replace_weights,
    TensorParallelNeuronModel,
)

# Free CPU text encoder (~24 GB)
del pipe.text_encoder
gc.collect()
print("Freed CPU text encoder")

# Load Neuron Gemma3 encoder from pre-sharded per-rank checkpoints
print(f"Loading Gemma3 encoder from {GEMMA3_COMPILE_DIR}...")
t0 = time.time()

models = []
for rank in range(TP_DEGREE):
    tr = time.time()
    rank_ckpt_path = os.path.join(GEMMA3_SHARDED_DIR, f"rank_{rank}.pt")
    ckpt = torch.load(rank_ckpt_path, weights_only=True)
    
    neff_path = os.path.join(GEMMA3_COMPILE_DIR, f"tp_{rank}.pt")
    with torch_neuronx.contexts.disable_nrt_load():
        traced_model = torch.jit.load(neff_path)
    
    replace_weights(traced_model, ckpt)
    models.append(traced_model)
    del ckpt
    gc.collect()
    print(f"  [rank {rank}] {time.time() - tr:.1f}s")

compiled_gemma3 = TensorParallelNeuronModel(models)
print(f"Gemma3 encoder loaded in {time.time() - t0:.1f}s")

In [ ]:
# Create a wrapper that mimics the HF text encoder interface
class NeuronTextEncoderOutput:
    def __init__(self, hidden_states):
        self.hidden_states = hidden_states

class NeuronTextEncoderWrapper:
    """Drop-in replacement for Gemma3ForConditionalGeneration."""
    def __init__(self, compiled_gemma3, dtype=torch.bfloat16):
        self.compiled_model = compiled_gemma3
        self.dtype = dtype
        self._device = torch.device("cpu")
        self.config = type("Config", (), {"output_hidden_states": True})()

    def __call__(self, input_ids=None, attention_mask=None, output_hidden_states=True, **kwargs):
        with torch.no_grad():
            # Neuron model returns (B, seq_len, hidden_size, num_layers+1)
            stacked = self.compiled_model(input_ids, attention_mask)
            num_states = stacked.shape[-1]
            hidden_states = tuple(stacked[:, :, :, i] for i in range(num_states))
        return NeuronTextEncoderOutput(hidden_states=hidden_states)

    def eval(self): return self
    def to(self, *args, **kwargs): return self

    @property
    def device(self): return self._device

pipe.text_encoder = NeuronTextEncoderWrapper(compiled_gemma3)
print("Text encoder swapped to Neuron Gemma3")

## 7. Generate Video + Audio

Run the full pipeline with the default parameters:
- `guidance_scale=4.0` (classifier-free guidance — the DiT runs twice per step)
- `max_sequence_length=1024`
- 8 denoising steps (distilled model)

The pipeline handles text encoding, denoising loop, VAE decoding, and vocoder automatically.

In [ ]:
print(f"Generating: {PROMPT[:80]}...")
print(f"{WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_STEPS} steps")

generator = torch.Generator(device="cpu").manual_seed(SEED)

t0 = time.time()
output = pipe(
    prompt=PROMPT,
    height=HEIGHT,
    width=WIDTH,
    num_frames=NUM_FRAMES,
    num_inference_steps=NUM_STEPS,
    generator=generator,
    output_type="pil",
)
gen_time = time.time() - t0
print(f"Generated in {gen_time:.1f}s")

## 8. Save and Display Results

In [ ]:
# Save frames and video
os.makedirs(OUTPUT_DIR, exist_ok=True)
frames = output.frames[0]

frames_dir = os.path.join(OUTPUT_DIR, "frames")
os.makedirs(frames_dir, exist_ok=True)
for i, frame in enumerate(frames):
    frame.save(os.path.join(frames_dir, f"frame_{i:04d}.png"))
print(f"Saved {len(frames)} frames to {frames_dir}/")

try:
    from diffusers.utils import export_to_video
    video_path = os.path.join(OUTPUT_DIR, "output.mp4")
    export_to_video(frames, video_path, fps=24)
    print(f"Video: {video_path}")
except Exception as e:
    print(f"Video export failed: {e}")

In [ ]:
# Display sample frames
from IPython.display import display
import PIL.Image

sample_indices = [0, len(frames) // 4, len(frames) // 2, 3 * len(frames) // 4, len(frames) - 1]
for idx in sample_indices:
    print(f"Frame {idx:04d}:")
    display(frames[idx].resize((256, 192)))  # Thumbnail for display

In [ ]:
# Save metadata
metadata = {
    "model": "Lightricks/LTX-2",
    "prompt": PROMPT,
    "resolution": f"{WIDTH}x{HEIGHT}",
    "num_frames": NUM_FRAMES,
    "num_steps": NUM_STEPS,
    "guidance_scale": 4.0,
    "max_sequence_length": 1024,
    "seed": SEED,
    "generation_time_s": gen_time,
    "text_encoder": f"Neuron Gemma3-12B (TP={TP_DEGREE})",
    "dit": f"Neuron LTX2 DiT 48 blocks (TP={TP_DEGREE})",
}

with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nGeneration time: {gen_time:.1f}s")
print(f"Output: {OUTPUT_DIR}")

## 9. Try Your Own Prompt

Change the prompt below and re-run to generate a different video.

In [ ]:
# Change this prompt to generate different content
my_prompt = "A cat sitting on a windowsill watches rain falling outside, with soft piano music playing."

generator = torch.Generator(device="cpu").manual_seed(123)

t0 = time.time()
output2 = pipe(
    prompt=my_prompt,
    height=HEIGHT,
    width=WIDTH,
    num_frames=NUM_FRAMES,
    num_inference_steps=NUM_STEPS,
    generator=generator,
    output_type="pil",
)
print(f"Generated in {time.time() - t0:.1f}s")

# Display first and last frame
from IPython.display import display
frames2 = output2.frames[0]
print(f"Frame 0:")
display(frames2[0].resize((256, 192)))
print(f"Frame {len(frames2)-1}:")
display(frames2[-1].resize((256, 192)))